In [ ]:
import folium
import geojson
import oracledb
from shapely.wkt import dumps, loads

# Definicja i test połączenia


In [ ]:
cs = oracledb.makedsn("dbmanage.lab.ii.agh.edu.pl", 1521, sid="DBMANAGE")

un = "student"
pw = "stu638dent"

connection = oracledb.connect(user=un, password=pw, dsn=cs)

In [ ]:
print(connection)

# Wybór schematu US_SPAT


In [ ]:
q = """ALTER SESSION SET CURRENT_SCHEMA = US_SPAT"""

cursor = connection.cursor()

cursor.execute(q)

In [ ]:
cursor = connection.cursor()

q = """select id, state from us_states"""
for row in cursor.execute(q):
    print(row)

In [ ]:
def OutputTypeHandler(cursor, name, defaultType, size, precision, scale):
    if defaultType == oracledb.CLOB:
        return cursor.var(oracledb.LONG_STRING, arraysize=cursor.arraysize)


connection.outputtypehandler = OutputTypeHandler

# Przykłady


# Florida & Texas


In [ ]:
# m = folium.Map()

m = folium.Map(location=[39, -98], zoom_start=4, tiles="OpenStreetMap")

q = """SELECT  sdo_util.to_wktgeometry(geom)
       FROM us_states
       WHERE state='Florida' or state = 'Texas'"""

# q = """SELECT  sdo_util.to_wktgeometry(geom)
#        FROM us_states """

r = loads(cursor.execute(q).fetchall())

st = {"fillColor": "blue", "color": "red"}

features = []

for row in r:
    g = geojson.Feature(geometry=row[0], properties={})
    features.append(g)


feature_collection = geojson.FeatureCollection(features)

folium.GeoJson(feature_collection, style_function=lambda x: st).add_to(m)

m

# Dodatkowa warstwa z drogami


In [ ]:
q = """SELECT  sdo_util.to_wktgeometry(geom)
       FROM us_interstates"""

r = loads(cursor.execute(q).fetchall())

st2 = {"color": "blue"}

features = []

for row in r:
    g = geojson.Feature(geometry=row[0], properties={})
    features.append(g)


feature_collection = geojson.FeatureCollection(features)

folium.GeoJson(feature_collection, style_function=lambda x: st2).add_to(m)

m

# Parki wewnątrz stanu Texas


In [ ]:
m = folium.Map(location=[39, -98], zoom_start=4, tiles="OpenStreetMap")

q = """SELECT  sdo_util.to_wktgeometry(p.geom)
FROM us_parks p, us_states s
WHERE s.state = 'Texas'
AND SDO_INSIDE (p.geom, s.geom ) = 'TRUE'"""

r = loads(cursor.execute(q).fetchall())

st = {"fillColor": "blue", "color": "red"}

features = []
for row in r:
    g = geojson.Feature(geometry=row[0], properties={})
    features.append(g)


feature_collection = geojson.FeatureCollection(features)
folium.GeoJson(feature_collection, style_function=lambda x: st).add_to(m)

# m.show_in_browser()
m